In [9]:
"""
Title: The Sensor Data Integrity
Author: A. Herimalala M.
Date  : 12-05-2026
Time Zone: Africa/Nairobi (time zone for Madagascar)
Jupyter Lab Version: 4.4.7
"""


""" 0. GEnerate the Dataset """
# General imports
import numpy as np

# Fix seed to debug

# cf: https://numpy.org/doc/2.1/reference/random/generated/numpy.random.normal.html
sample_data = np.random.normal(loc=50, scale=10, size=1000)

# Corrupt 50 items choosed randomly 
to_corrupt = np.random.randint(0, 1000, size=50)
sample_data[to_corrupt] = np.nan

# Data Shape
shape = sample_data.shape # 1000 cells
print(f"Shape: ", shape)

# DType 
print(sample_data.dtype) # float64

# Anomaly threshold
K = 1.5
K

Shape:  (1000,)
float64


1.5

In [10]:
""" Data Cleaning 
    use np.nanmedian
"""
# Here we don't know indices dynamicaly
# sample_data[corrupted_indices] = 10.0 

# Get median of data with NaN values and after replace NaN values with its value
# cf:https://numpy.org/doc/stable/reference/generated/numpy.nanmedian.html
median_value = np.nanmedian(sample_data)
global_std_before_filled_nan = np.nanstd(sample_data)

# Replace NaN values with median_value 
sample_data[np.isnan(sample_data)] = median_value

In [11]:

""" Reshaping the data into (200, 5)"""
sample_data = sample_data.reshape((200, 5))

In [12]:
""" Statistical profiling 
    safe function for std (cf: https://numpy.org/doc/stable/reference/generated/numpy.nanstd.html)
    safe function for mean (cf: https://numpy.org/doc/stable/reference/generated/numpy.nanmean.html)
"""
# Todo: Use safe functions to calculate mean and std 
# Choose axis = 1 to capture each row
# mean_values = np.mean(sample_data, axis=1) unsafe 
# std_values = np.std(sample_data, axis=1) unsafe
mean_values = np.nanmean(sample_data, axis=1)
std_values = np.nanstd(sample_data, axis=1)

In [14]:

""" Anomaly Detection 
    Here, we must use global std because 
    use local std_values with global global_mean is nosense (statistically)

    at work i compute statistics of pointcloud data 
    then the Point Cloud Technician told me to capture only data between the range [mean_or_median - k * std, mean_or_median + k * std]
    where k is sometimes 2.5
    because that's avoid extremums data to influence the reality(very low/high data)
    Since then, i have preferred using the median.
"""
# Disable exponential displaying
np.set_printoptions(suppress=True, precision=5)
global_std_after_filled_nan = np.nanstd(sample_data)

# If there is significant difference between these two variable, may be it's not recomma
print(f"Std before fill NaN with median: {global_std_before_filled_nan}")
print(f"Std after fill NaN with median: {global_std_after_filled_nan}")

# Valid datas must be in the range [global_mean - 1.5 * global_std, global_mean + 1.5 * global_std] 
global_mean = np.mean(mean_values.ravel())
print("Global Mean: ", global_mean)

# retrieve them with boolean indexing
anomalies_mask = (mean_values > global_mean + 1.5 * global_std_after_filled_nan) | (mean_values < global_mean - 1.5 * global_std_after_filled_nan)
anomalies_data = sample_data[anomalies_mask]
print(anomalies_data) # For this case there's no anomalies data !!!!?? 

Std before fill NaN with median: 9.815183047904899
Std after fill NaN with median: 9.571722534878138
Global Mean:  49.970741732586355
[]


In [15]:

""" Normalization """
# Disable exponential displaying
np.set_printoptions(suppress=True, precision=5)
# Todo: What is the difference between using global_min & global_max instead of using min & max per row ???
# Todo: Scales all values to fit [0, 1] ranges

""" 1- USE GLOBAL MIN & MAX 
    wE USE Global min & max if we aim to detect anomalie
    in data and visualize rows with abnormal tandency
"""
min_values = np.min(sample_data, axis=1)
global_min = np.min(min_values)

max_values = np.max(sample_data, axis=1)
global_max = np.max(max_values)
print(f"Global MIn: {global_min}, Global Max: {global_max}")
global_sample_normalized = (sample_data - global_min)/(global_max - global_min) 

""" 2- USE MIN & MAX PER ROW 
    We use per row min & max if we aim 
    to analyze forms of the data.
"""
min_values = min_values.reshape((200, 1))
max_values = max_values.reshape((200, 1))
per_row_sample_normalized = (sample_data - min_values)/(max_values - min_values)

Global MIn: 11.53896885702244, Global Max: 83.86941051432436
